In [23]:
# Importy
import pandas as pd
import json
import os
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

In [24]:
# Dane IFC
ifc_path = "../data/testing.csv"
ifc_df = pd.read_csv(ifc_path)
print(f"Wczytano {len(ifc_df)} obiektów IFC")

Wczytano 887 obiektów IFC


In [25]:
# Dane z bSDD
with open("../data/bsdd_cleared.json", encoding="utf-8") as f:
    bsdd_raw = json.load(f)
    
    bsdd_rows = []
    for group in bsdd_raw:
        for cls in group["classes"]:
            bsdd_rows.append({
            "dictionary_name": group["dictionary_name"],
            "class_code": cls.get("class_code", ""),
            "class_name": cls.get("class_name", ""),
            "class_description": cls.get("class_description", "")
            })

bsdd_df = pd.DataFrame(bsdd_rows)
print(f"Wczytano {len(bsdd_df)} klas bSDD")

Wczytano 3031 klas bSDD


In [4]:
# Mapowanie IFC
ifc_to_keywords = {
"IFCWALL": ["wall"],
"IFCDOOR": ["door"],
"IFCWINDOW": ["window"],
"IFCSLAB": ["slab", "floor", "roof"],
"IFCSTAIR": ["stair", "step"],
"IFCCOLUMN": ["column", "pillar"],
"IFCBEAM": ["beam"],
"IFCFURNISHINGELEMENT": ["furniture", "chair", "table", "shelf"],
"IFCRAILING": ["railing", "barrier"],
}

In [32]:
# Model
model = SentenceTransformer("../models/fewshot_finetuned")
# model = SentenceTransformer("../models/parsed_tsdae_model")

In [33]:
# Przygotowanie danych
bsdd_df["full_text"] = (
bsdd_df["class_code"].astype(str)
+ " — "
+ bsdd_df["class_name"].fillna("")
+ " — "
+ bsdd_df["class_description"].fillna("")
)

In [34]:
# Klasyfikacja
from sentence_transformers import util
from tqdm import tqdm

results = []

for i, row in tqdm(ifc_df.iterrows(), total=len(ifc_df)):
    ifc_type = row.get("IfcType", "").upper()
    ifc_text = str(row["Text"])
    ifc_name = str(row.get("Name", ""))

    # Słowa kluczowe dla danego typu IFC
    keywords = ifc_to_keywords.get(ifc_type.upper(), [])
    
    # Filtrowanie klas bSDD na podstawie słów kluczowych
    filtered_bsdd = bsdd_df[
        bsdd_df["class_name"].str.lower().str.contains("|".join(keywords), na=False)
        | bsdd_df["class_description"].str.lower().str.contains("|".join(keywords), na=False)
    ]
    
    if filtered_bsdd.empty: 
        continue
    
    # Embeddingi i podobieństwa
    ifc_embedding = model.encode([ifc_name + " " + ifc_text], convert_to_tensor=True)
    bsdd_embeddings = model.encode(filtered_bsdd["full_text"].tolist(), convert_to_tensor=True)
    
    similarities = util.cos_sim(ifc_embedding, bsdd_embeddings)[0]
    best_idx = similarities.argmax().item()
    best_score = similarities[best_idx].item()
    
    best_class = filtered_bsdd.iloc[best_idx]
    
    results.append({
        "GlobalId": row["GlobalId"],
        "IfcType": ifc_type,
        "Name": ifc_name,
        "Opis_IFC": ifc_text[:300],
        "Kod_bSDD": best_class["class_code"],
        "Nazwa_klasy_bSDD": best_class["class_name"],
        "Słownik": best_class["dictionary_name"],
        "Podobieństwo": round(best_score, 4)
    })

  0%|                                                                                          | 0/887 [00:00<?, ?it/s]C:\Users\chlip\anaconda3\envs\bim-nlp\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████████████████████████████████████████████████████████████████████████| 887/887 [2:47:38<00:00, 11.34s/it]


In [35]:
# Wyniki
results_df = pd.DataFrame(results)
results_df.to_csv("../results/fewshot_testing.csv", index=False)
print("Zapisano wynik do fewshot_testing.csv")

Zapisano wynik do fewshot_testing.csv


In [36]:
# Wyświetlanie bezpośrednio pod komórką
display(results_df)

,GlobalId,IfcType,Name,Opis_IFC,Kod_bSDD,Nazwa_klasy_bSDD,Słownik,Podobieństwo
0,0a3v3dJi10mxIqGCSrYdxN,,0001,. Description: ViewDefinition [CoordinationVie...,Ac_05_10_67,Project brief and objectives submission,Uniclass,0.8197
1,0a3v3dJi10mxIqGCSrYdxL,,Default,. Category: Project Information; Project: 0001...,Ac_15_30_90,Topographical surveying,Uniclass,0.7744
2,0a3v3dJi10mxIqGCSrYdxM,,nan,. Address lines: Enter address here; Country: ...,Ac_15_30_90,Topographical surveying,Uniclass,0.7629
3,0a3v3dJi10mxIqGCVATO0G,,Street level,. Name: Street level; AboveGround: 9.881312916...,Ac_15_30_90,Topographical surveying,Uniclass,0.7482
4,0ZRQUHwuv8SOaxY18jH6Mu,,Floor:Generic 150mm - Filled:348347,. Roughness: 914.4; Category: Floors; Referenc...,Ac_15_50_73,Roof surveying,Uniclass,0.7621
...,...,...,...,...,...,...,...,...
882,2BV8ty4rH4pOWhxXNZufaw,,Rectangular Mullion:50 x 150mm:577379,. ProfileName: 50 x 150mm; XDim: 150; YDim: 50...,Ac_15_50_73,Roof surveying,Uniclass,0.7757
883,18J$4BjujFTBNZhLt6iXLe,,Sink - Bathroom (4):660 x 560mm:560746,. Reference: 660 x 560mm; Manufacturer: Revit;...,Ac_30_60_96,Washing up,Uniclass,0.7172
884,15b09kJ7D9nPemCyFAchF6,,Railing:900mm Pipe:552562,. Category: Railings; Reference: 900mm Pipe; H...,Ac_15_30_90,Topographical surveying,Uniclass,0.7628
885,0a3v3dJi10mxIqGCVASD4d,,Roof,. Name: Roof; AboveGround: 9.88131291682493E-3...,Ac_15_50_73,Roof surveying,Uniclass,0.7671
